# Koopmans theorem

Import all necessary libraries for this notebook.

In [51]:
import pymolpro
from pymolpro import ASEMolpro
from ase.visualize import view
from ase.build import molecule
from ase.optimize import BFGS
from ase import Atoms
from ase.io import read, write
from pyscf import tools
import nglview as nv
from ase.io import read
from ase import units

Create and optimize the molecule N2. Use the Hartree Fock method and the basis set cc-pVDZ. The convergence criterion can be set to 0.0001 eV/Angstrom. Then, save it as an xyz file.

In [4]:
N2 = molecule("N2")
N2.calc = ASEMolpro(ansatz='HF/cc-pVDZ', density_fitting=True)
with BFGS(N2, trajectory='n2.traj') as opt:
    opt.run(fmax=0.0001)
write("N2_opt.xyz", N2)

      Step     Time          Energy          fmax
BFGS:    0 13:40:34    -2964.581306        8.661175
BFGS:    1 13:40:39    -2958.894323       75.999402
BFGS:    2 13:40:43    -2964.754371        4.898446
BFGS:    3 13:40:47    -2964.805110        2.610216
BFGS:    4 13:40:52    -2964.823436        0.271783
BFGS:    5 13:40:56    -2964.823623        0.013035
BFGS:    6 13:41:00    -2964.823624        0.000061


Now, we want to create a molden file for the optimized N2 molecule. This file contains the information about the orbitals and their energies. We will use this file later to plot the orbitals.

Unfortunately, the molden file cannot be created directly using ase. Thus, we need to write the input to molpro directly, which is shown in the example below. The geometry of the molecule is read from the xyz file we created in the previous step, for which an abolute path needs to be given.

In [5]:
# define a project name, which is used for the folder where the results are stored. 
n2_project = "N2"

In [12]:
# 1. Generate a pymolpro object
from httpx import put


project = pymolpro.Project(n2_project)

# 2. Write the input file for molpro. The input file is a string.
# Explanation of the following lines of code:
#    enter the absolute path for the geometry xyz file
#    enter the correct basis set (pp-pvdz)
#    hf stands for hartree fock
#    wf stands for wave function
#    enter the correct number of electrons for the molecule
#    enter the correct symmetry (1 in our case)
#    enter the correct number of unpaired electrons
#    put,molden,n2.molden: tells molpro to write the orbitals in molden format

molpro_input = """
orient,planexz,mass;
gprint,orbitals;

geometry=/home/CompChem3/Documents/Compchem/6/N2_opt.xyz

basis=cc-pvdz
{hf;wf,14,1,0}

put,molden,n2.molden
"""

# 3. Write the input file to the pymolpro project
project.write_input(molpro_input)

# 4. Run the project. This will execute molpro and write the output file.
project.run()


Molpro creates a folder structure like this:
- the project called like n2_project is saved in the folder of the same name
- all calculations that are performed with thte same project are saved in the same folder
- the results are saved in the folder called run
- the first run is called project_name_1.molpro, the second run is called project_name_2.molpro, and so on
- the file we are interested is called "n2.molden"

Create a string that contains the path to the molden file.

In [13]:
run = 5
n2_molden_file = n2_project + ".molpro/run/" + n2_project + "_" + str(run) + ".molpro/" + "n2.molden"
print(n2_molden_file)

N2.molpro/run/N2_5.molpro/n2.molden


Using tools from the library pyscf, we can read the molden file and extract the orbital information, like the orbital energies, orbital occupations, irred. lables, and the coefficients of the orbitals.

In [14]:
n2_mol, n2_mo_energy, n2_mo_coeff, n2_mo_occ, n2_irred_labels, n2_spins = tools.molden.load(n2_molden_file)

Unknown section MOLPRO VARIABLES


Take a look at the variables containing the energies, coefficients, occupations, irred. lables and the spins of the orbitals.

In [16]:
print(n2_mo_energy)


[-15.68   -15.6761  -1.4864  -0.7674  -0.6279  -0.617   -0.617    0.1867
   0.1867   0.6007   0.8126   0.8662   0.8662   0.9948   1.0559   1.0559
   1.1762   1.6653   1.7508   1.7508   1.9035   1.9035   2.3132   2.3132
   2.8962   3.0195   3.0195   3.3333]


To plot the molecules, the nglviewer can be used. It can plot so called cube files, which contain the information about the orbitals in a 3D grid. The cube file can be created using the molden file we created in the previously. Each cube file contains only the information of one orbital, thus we need to define the index of the orbital we want to plot when creating the cube file. Be aware, the index starts with 0.

In [19]:
# define index you want to visualize
for orbital_index in range(n2_mo_coeff.shape[1]):
    tools.cubegen.orbital(n2_mol, f'N2_{orbital_index}.cube', n2_mo_coeff[:, orbital_index])

28


Now, ase can read the cube file and we can plot the orbital using nglview. The isovalue defines the value of the wavefunction at which the surface is plotted. You can play around with this value to see how it affects the plot.

In [26]:
atoms = read('N2_2.cube')

view = nv.show_ase(atoms)

view.add_component('N2_2.cube')
view.clear_representations()
view.add_ball_and_stick()
view.add_surface(component=1, isolevelType="value", isolevel=0.05, color="blue", opacity=0.6, side="front")
view.add_surface(component=1, isolevelType="value", isolevel=-0.05, color="red", opacity=0.6, side="front")

view

NGLWidget()

Noww, we will do the same with the cationic N2 molecule.

In [49]:
# create, optimize and save the geometry of n2+
# don't forget to set the charge to +1
N2p = molecule("N2")
N2p.calc = ASEMolpro(ansatz='HF/cc-pVDZ', density_fitting=True, charge=1)
with BFGS(N2p, trajectory='n2+.traj') as opt:
    opt.run(fmax=0.0001)
write("N2+_opt.xyz", N2p)

      Step     Time          Energy          fmax
BFGS:    0 14:27:59    -2948.790774        6.014969
BFGS:    1 14:28:04    -2946.837748       36.871211
BFGS:    2 14:28:09    -2948.896005        2.614502
BFGS:    3 14:28:14    -2948.914030        1.051848
BFGS:    4 14:28:19    -2948.917311        0.064124
BFGS:    5 14:28:24    -2948.917323        0.001441
BFGS:    6 14:28:28    -2948.917323        0.000002


Create a project and start the molden calculation. It sould yield the molden file for the n2+ molecule.

In [29]:
# TODO write code
# make sure you use the correct number for electrons in the molecule and unparied electrons
# to set the charge, enter after "orient,planexz,mass;" a new line containing "charge = +1"

from httpx import put

n2p_project = "N2+"
project = pymolpro.Project(n2p_project)

molpro_input = """
orient,planexz,mass;
charge=+1
gprint,orbitals;

geometry=/home/CompChem3/Documents/Compchem/6/N2+_opt.xyz

basis=cc-pvdz
{hf;wf,13,1,1}

put,molden,n2p.molden
"""

project.write_input(molpro_input)
project.run()


Then, make a string with the path to the molden file, load it and create the cube file for the orbital you want to plot, e.g. the HOMO. you can check which orbital index belongs to the HOMO by looking at orbital occupations.

In [32]:
run = 1
orbital_index = 6

n2p_molden_file = n2p_project + ".molpro/run/" + n2p_project + "_" + str(run) + ".molpro/" + "n2p.molden"
n2p_mol, n2p_mo_energy, n2p_mo_coeff, n2p_mo_occ, n2p_irred_labels, n2p_spins = tools.molden.load(n2p_molden_file)
tools.cubegen.orbital(n2p_mol, f'N2+_{orbital_index}.cube', n2p_mo_coeff[:, orbital_index])

Unknown section MOLPRO VARIABLES


array([[[0.00042473, 0.0004731 , 0.00052426, ..., 0.00052426,
         0.0004731 , 0.00042473],
        [0.00046908, 0.00052252, 0.00057904, ..., 0.00057904,
         0.00052252, 0.00046908],
        [0.00051675, 0.00057563, 0.00063792, ..., 0.00063792,
         0.00057563, 0.00051675],
        ...,
        [0.00051675, 0.00057563, 0.00063792, ..., 0.00063792,
         0.00057563, 0.00051675],
        [0.00046908, 0.00052252, 0.00057904, ..., 0.00057904,
         0.00052252, 0.00046908],
        [0.00042473, 0.0004731 , 0.00052426, ..., 0.00052426,
         0.0004731 , 0.00042473]],

       [[0.00046908, 0.00052252, 0.00057904, ..., 0.00057904,
         0.00052252, 0.00046908],
        [0.00051806, 0.0005771 , 0.00063955, ..., 0.00063955,
         0.0005771 , 0.00051806],
        [0.00057071, 0.00063577, 0.00070458, ..., 0.00070458,
         0.00063577, 0.00057071],
        ...,
        [0.00057071, 0.00063577, 0.00070458, ..., 0.00070458,
         0.00063577, 0.00057071],
        [0.0

In the following, we will often plot molecules. Thus, we will create a function that plots the molecule using nglview. This function takes the path to the cube file as input and plots the orbital.

In [38]:
def plot_orbitals(cube_file: str):
    atoms = read(cube_file)

    view = nv.show_ase(atoms)

    view.add_component(cube_file)
    view.clear_representations()
    view.add_ball_and_stick()
    view.add_surface(component=1, isolevelType="value", isolevel=0.05, color="blue", opacity=0.6, side="front")
    view.add_surface(component=1, isolevelType="value", isolevel=-0.05, color="red", opacity=0.6, side="front")

    return view

In [1]:
plot_orbitals("N2+_6.cube")

NameError: name 'plot_orbitals' is not defined

Calculate the ionization energy in two different ways: using the Koopmans theorem and using the total energy difference between the neutral molecule and the cation. Compare the results and discuss the differences.

In [48]:
# Koopmans Theorem
print(f"Ionization energy (Koopman): {-n2p_mo_energy[6]:.3f}eV")

# Total energy difference
print(f"Ionization energy (Energy Difference): {N2p.get_total_energy() - N2.get_total_energy():.3f}eV")

Ionization energy (Koopman): 1.132eV
Ionization energy (Energy Difference): 15.906eV


# Orbitals


To save the orbital information, we will use the CUBE format.

More information in https://www.molpro.net/manual/doku.php?id=dump_density_or_orbital_values_cube&s[]=cube



## Canonical orbital



Create, optimize, view and save the furan molecule.

In [50]:
furan = molecule("C4H4O")
furan.calc = ASEMolpro(ansatz='HF/cc-pVDZ', density_fitting=True)
with BFGS(furan, trajectory='furan.traj') as opt:
    opt.run(fmax=0.0001)
write("furan_opt.xyz", furan)

      Step     Time          Energy          fmax
BFGS:    0 14:28:48    -6221.662960        1.445107
BFGS:    1 14:28:56    -6221.701084        1.028242
BFGS:    2 14:29:06    -6221.715922        0.315074
BFGS:    3 14:29:17    -6221.718507        0.167444
BFGS:    4 14:29:26    -6221.719967        0.073973
BFGS:    5 14:29:35    -6221.720504        0.068402
BFGS:    6 14:29:44    -6221.720724        0.066278
BFGS:    7 14:29:54    -6221.721161        0.049220
BFGS:    8 14:30:03    -6221.721347        0.040377
BFGS:    9 14:30:13    -6221.721448        0.019046
BFGS:   10 14:30:22    -6221.721469        0.007909
BFGS:   11 14:30:32    -6221.721472        0.002115
BFGS:   12 14:30:54    -6221.721472        0.001553
BFGS:   13 14:31:03    -6221.721472        0.001269
BFGS:   14 14:31:12    -6221.721472        0.001212
BFGS:   15 14:31:20    -6221.721473        0.000586
BFGS:   16 14:31:30    -6221.721473        0.000139
BFGS:   17 14:31:41    -6221.721473        0.000011


In [52]:
view(furan, viewer="x3d")

Use molpro to create a molden file for the optimized furan molecule. This file contains the canonical orbitals, which are the same type of orbitals as above.

In [54]:
# TODO create molden and cube files as above. The default is canonical orbitals.
# instead of using {hf;wf,...} in the input file, use only rhf

from httpx import put

furan_project = "furan"
project = pymolpro.Project(furan_project)

molpro_input = """
orient,planexz,mass;
gprint,orbitals;

geometry=/home/CompChem3/Documents/Compchem/6/furan_opt.xyz

basis=cc-pvdz
{rhf;wf,36,1,0}

put,molden,furan.molden
"""

project.write_input(molpro_input)
project.run()

Read in the molden file as above.

In [56]:
run = 2

furan_molden_file = furan_project + ".molpro/run/" + furan_project + "_" + str(run) + ".molpro/" + "furan.molden"
furan_mol, furan_mo_energy, furan_mo_coeff, furan_mo_occ, furan_irred_labels, furan_spins = tools.molden.load(furan_molden_file)
print(furan_mo_occ)

Unknown section MOLPRO VARIABLES


[2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Define the indices of the orbitals you want to plot. Which one is the HOMO and the LUMO?

In [57]:
homo_index = 17
lumo_index = 18
core_index1 = 5


Write the cube files for all the indices you defined. Make sure to lable the files in an understandable way, so you know later which file corresponds to which types of orbital, molecule, method, etc.

In [58]:
tools.cubegen.orbital(furan_mol, f'furan_{homo_index}.cube', furan_mo_coeff[:, homo_index])
tools.cubegen.orbital(furan_mol, f'furan_{lumo_index}.cube', furan_mo_coeff[:, lumo_index])
tools.cubegen.orbital(furan_mol, f'furan_{core_index1}.cube', furan_mo_coeff[:, core_index1])

array([[[ 5.48925344e-06,  6.01025851e-06,  6.54141598e-06, ...,
         -1.56009481e-05, -1.41647485e-05, -1.27773358e-05],
        [ 5.78560091e-06,  6.33364559e-06,  6.89214851e-06, ...,
         -1.63776852e-05, -1.48779621e-05, -1.34271868e-05],
        [ 6.08954705e-06,  6.66525290e-06,  7.25171543e-06, ...,
         -1.71689938e-05, -1.56053191e-05, -1.40905192e-05],
        ...,
        [ 6.08954705e-06,  6.66525290e-06,  7.25171543e-06, ...,
         -1.71689938e-05, -1.56053191e-05, -1.40905192e-05],
        [ 5.78560091e-06,  6.33364559e-06,  6.89214851e-06, ...,
         -1.63776852e-05, -1.48779621e-05, -1.34271868e-05],
        [ 5.48925344e-06,  6.01025851e-06,  6.54141598e-06, ...,
         -1.56009481e-05, -1.41647485e-05, -1.27773358e-05]],

       [[ 6.42712296e-06,  7.02842671e-06,  7.63982591e-06, ...,
         -1.66870806e-05, -1.52191183e-05, -1.37849928e-05],
        [ 6.77239214e-06,  7.40458711e-06,  8.04711154e-06, ...,
         -1.74954122e-05, -1.59673484e

Plot the orbitals.

In [59]:
plot_orbitals("furan_17.cube")

NGLWidget()

## Localized orbitals

More information related with the localized orbitals:
https://www.molpro.net/manual/doku.php?id=orbital_localization&s[]=locali

Now, we want to create the molden file to plot the localized molecules as well. We only need to change the input to the molpro-project, the rest of the steps are the same as for the canonical orbitals.

Input that needs to be changed:
- add 'nosym' as first line
- add '{locali,boys}' to convert the canonical orbitals to localized orbitals
- add 'record=2100.2' after molden, so molpro knows which orbitals to use to create the molden file

In [61]:
# create project name
localized_project = "localized_furan"

In [62]:
project = pymolpro.Project(localized_project)

molpro_input = """
nosym
orient,planexz,mass;
gprint,orbitals;

geometry=/home/CompChem3/Documents/Compchem/6/furan_opt.xyz

basis=cc-pvdz
{rhf;wf,36,1,0}

{locali,boys}

put,molden,furan_loc.molden,record=2100.2
"""

project.write_input(molpro_input)
project.run()

Read in the molden file and create the cube files for the same indices as you defined above for the canonical orbitals.

In [64]:
run = 1

furan_loc_molden_file = localized_project + ".molpro/run/" + localized_project + "_" + str(run) + ".molpro/" + "furan_loc.molden"
furan_loc_mol, furan_loc_mo_energy, furan_loc_mo_coeff, furan_loc_mo_occ, furan_loc_irred_labels, furan_loc_spins = tools.molden.load(furan_loc_molden_file)

tools.cubegen.orbital(furan_loc_mol, f'furan_loc_{homo_index}.cube', furan_loc_mo_coeff[:, homo_index])
tools.cubegen.orbital(furan_loc_mol, f'furan_loc_{lumo_index}.cube', furan_loc_mo_coeff[:, lumo_index])
tools.cubegen.orbital(furan_loc_mol, f'furan_loc_{core_index1}.cube', furan_loc_mo_coeff[:, core_index1])

Unknown section MOLPRO VARIABLES


array([[[-4.11899930e-05, -4.55043266e-05, -4.99689952e-05, ...,
          9.07976944e-05,  7.99255919e-05,  7.00238589e-05],
        [-4.35086671e-05, -4.80638938e-05, -5.27771101e-05, ...,
          9.60838720e-05,  8.45681771e-05,  7.40825370e-05],
        [-4.58936733e-05, -5.06965957e-05, -5.56653401e-05, ...,
          1.01534771e-04,  8.93545542e-05,  7.82662079e-05],
        ...,
        [-4.66598359e-05, -5.16282620e-05, -5.67901479e-05, ...,
          1.06373995e-04,  9.33570858e-05,  8.15550928e-05],
        [-4.42446506e-05, -4.89588619e-05, -5.38576118e-05, ...,
          1.00732297e-04,  8.84129043e-05,  7.72417586e-05],
        [-4.18952739e-05, -4.63619598e-05, -5.10044222e-05, ...,
          9.52520342e-05,  8.36097954e-05,  7.30511822e-05]],

       [[-4.90395623e-05, -5.41603664e-05, -5.94542309e-05, ...,
          1.02956408e-04,  9.05965914e-05,  7.93478244e-05],
        [-5.18003177e-05, -5.72069449e-05, -6.27953613e-05, ...,
          1.08960215e-04,  9.58667695e

Plot the localized and the canoninical orbitals to compare them.

In [65]:
plot_orbitals("furan_loc_17.cube")

NGLWidget()

Find out by plotting different orbitals which orbitals are core orbitals, which represent sigma and which pi bonds. Save for each of them an example of the canonical and the localized orbital as png file to include them in the report later.

In [76]:
core_index = 4

tools.cubegen.orbital(furan_mol, f'furan_{core_index}.cube', furan_mo_coeff[:, core_index])
tools.cubegen.orbital(furan_loc_mol, f'furan_loc_{core_index}.cube', furan_loc_mo_coeff[:, core_index])

plot_orbitals(f"furan_{core_index}.cube")

NGLWidget()

In [77]:
plot_orbitals(f"furan_loc_{core_index}.cube")

NGLWidget()